# Analisis Sentimen Komentar Traveloka di X
### Menggunakan Metode Multinomial Naive Bayes

M. Farhan Nailul Reza — 2110651076  
Teknik Informatika, Universitas Muhammadiyah Jember

---

Jalankan sel dari atas ke bawah secara berurutan.

| Sel | Tahap | Waktu |
|---|---|---|
| 1–2 | Siapkan proyek & install | 5 menit |
| 3 | Upload cookie X | 2 menit |
| 4 | Scraping | 30–60 menit |
| 5 | Preprocessing + penyaringan | 3 menit |
| 6 | Pra-anotasi lexicon | 1 menit |
| 7 | Ambil sampel validasi | 1 menit |
| 8 | **Pelabelan manual (Anda)** | 2–4 jam |
| 9 | Cohen's Kappa | 1 menit |
| 10 | Bangun dataset label manusia | 1 menit |
| 11 | Training & evaluasi | 3 menit |

> ⚠️ **Colab memutus koneksi saat menganggur.** Selama scraping jangan tutup
> tab. Bila terputus, hasil sementara tetap tersimpan — cukup jalankan ulang
> sel scraping, lalu gabungkan dengan `--merge` di sel preprocessing.

---
## ⚠️ Sebelum Membagikan Notebook Ini

**1. Bagikan link NOTEBOOK saja, bukan folder Drive.**  
Folder proyek di Drive memuat `session/cookies.json` — setara password akun X.

**2. Bersihkan output dulu:** **Edit → Clear all outputs**, lalu simpan.

**3. Izin berbagi = *Viewer*, bukan *Editor*.**  
Dengan izin Editor, orang lain dapat menyisipkan kode yang mengirim cookie Anda.

### Untuk pengguna yang menerima notebook ini

**File → Save a copy in Drive**, lalu jalankan dari sel 1. Anda memakai akun X
dan Drive sendiri; cookie Anda tidak terlihat oleh siapa pun.

> Scraping otomatis melanggar Terms of Service X. Gunakan hanya untuk
> keperluan akademis.

## 1. Siapkan Proyek

Mengunduh kode dari repositori dan menghubungkan Drive agar hasil tidak
hilang saat runtime di-reset. Tidak ada yang perlu di-upload manual.

In [ ]:
import os
from google.colab import drive

REPO  = 'https://github.com/fahmikemal/traveloka-scraper.git'
KERJA = '/content/drive/MyDrive/traveloka-sentimen'

drive.mount('/content/drive')

if os.path.exists(os.path.join(KERJA, 'main.py')):
    print('Proyek sudah ada, mengambil pembaruan...')
    os.chdir(KERJA)
    !git pull --ff-only 2>&1 | tail -3
else:
    print('Mengunduh proyek...')
    !git clone --depth 1 $REPO /content/_src 2>&1 | tail -2
    os.makedirs(KERJA, exist_ok=True)
    !cp -r /content/_src/. $KERJA/
    os.chdir(KERJA)

for d in ['data/raw','data/processed','data/labeled','data/gold',
          'models','results','session']:
    os.makedirs(d, exist_ok=True)

print('\nDirektori :', os.getcwd())
print('main.py   :', 'OK' if os.path.exists('main.py') else 'HILANG')
print('lexicon   :', 'OK' if os.path.exists('lexicon/positive.tsv') else 'HILANG')
print('kamus alay:', 'OK' if os.path.exists('lexicon/kamus_alay.tsv') else 'HILANG')

## 2. Install Library

Ulangi bila runtime di-restart.

In [ ]:
!pip install -q -r requirements.txt
!playwright install-deps chromium 2>/dev/null | tail -1
!playwright install chromium 2>&1 | tail -1

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

import importlib.metadata as md
print()
for p in ['playwright','pandas','scikit-learn','nltk','PySastrawi','langdetect','scipy']:
    try:    print(f'  {p:14} {md.version(p)}')
    except: print(f'  {p:14} BELUM TERPASANG')

## 3. Upload Cookie Session X

**Lakukan di browser Anda sendiri:**

1. Pasang ekstensi **Cookie-Editor** di Chrome/Edge
2. Buka <https://x.com> dan login
3. Klik ikon Cookie-Editor → **Export** (tersalin ke clipboard)
4. Buat file `cookies.json` di komputer, paste, simpan
5. Jalankan sel di bawah, pilih file tersebut

In [ ]:
import os, json, shutil
from google.colab import files

os.makedirs('session', exist_ok=True)
print('Pilih file cookies.json Anda...')
up = files.upload()

shutil.move(list(up.keys())[0], 'session/cookies.json')
os.chmod('session/cookies.json', 0o600)

d = json.load(open('session/cookies.json', encoding='utf-8'))
nm = {c.get('name') for c in d}
print(f'\nTersimpan: {len(d)} cookie')
for w in ('auth_token', 'ct0'):
    print(f'  {w:12}: {"ADA" if w in nm else "TIDAK ADA -> cookie tidak lengkap"}')
print('\nFile ini tersimpan di Drive ANDA. Jangan bagikan foldernya.')

## 4. Scraping

33 kata kunci layanan Traveloka, query memakai `-from:traveloka` untuk
membuang balasan customer service resmi.

**Progres disimpan setiap query selesai**, jadi terputusnya Colab tidak
menghilangkan hasil.

> Naikkan `--max` untuk data lebih banyak. Target ≥500 dokumen final;
> perkirakan penyusutan sekitar 60% oleh penyaringan di sel berikutnya.

In [ ]:
!python main.py --scrape --max 100 --headless

In [ ]:
import pandas as pd, glob
f = sorted(glob.glob('data/raw/*.csv'))
total = 0
for x in f:
    d = pd.read_csv(x, encoding='utf-8-sig'); total += len(d)
    print(f'{x}: {len(d)} tweet | {d["date"].min()[:10]} s/d {d["date"].max()[:10]}')
print(f'\nTOTAL sebelum dedup: {total}')
if f: display(pd.read_csv(f[-1], encoding='utf-8-sig').head(3))

## 5. Preprocessing + Penyaringan

Delapan tahap:

1. Cleaning (URL, mention, hashtag, angka, tanda baca)
2. Case folding
3. **Filter bahasa** — langdetect, hanya `id` confidence ≥ 0,70
4. **Filter relevansi** — buang dokumen yang tidak menyebut Traveloka
5. **Filter akun korporat** — buang iklan bank/telko
6. **Batas dokumen per penulis** — cegah satu akun mendominasi
7. **Normalisasi slang** — Kamus Alay, 15.115 pemetaan
8. Tokenisasi → stopword (negasi dipertahankan) → stemming Sastrawi

`--merge` menggabungkan semua sesi scraping di `data/raw/`.

In [ ]:
!python main.py --preprocess --merge --max-per-author 3

## 6. Pra-anotasi Lexicon (InSet)

> **PENTING.** Label ini **bukan** sumber label penelitian. Pada pengujian
> terhadap 588 dokumen berlabel manusia, pelabelan lexicon memperoleh
> **Cohen's Kappa 0,055** — praktis tidak lebih baik daripada menebak.
>
> Fungsinya di sini hanya dua: **objek yang dievaluasi** (sel 9), dan
> **usulan** untuk mempercepat pelabelan manual (sel 8).

In [ ]:
!python main.py --autolabel

## 7. Ambil Sampel untuk Validasi

Sampel acak berstrata — proporsi kelasnya mengikuti dataset penuh.

- **200 sampel** cukup untuk mengukur Kappa (praktik standar 100–300)
- Untuk **melatih model pada label manusia**, labeli seluruh dataset

In [ ]:
!python main.py --sample-gold 200

## 8. Pelabelan Manual — Dikerjakan Anda Sendiri

Ini bagian yang **tidak dapat diotomatiskan**. Cohen's Kappa mengukur
kesepakatan antara mesin dan **penilai manusia yang independen**; bila diisi
otomatis, angkanya tidak bermakna.

### Cara di Colab: Google Sheets

1. Buka Drive → `traveloka-sentimen/data/gold/gold_sample_*.csv`
2. Klik kanan → **Open with → Google Sheets**
3. Isi kolom **`label_manual`**
4. **File → Download → Comma Separated Values (.csv)**
5. Upload kembali ke folder `data/gold/` (timpa yang lama)

> Sembunyikan kolom `label_auto` dulu agar penilaian Anda tidak terpengaruh.

### Cara di komputer sendiri (lebih cepat)

```bash
python label_manual.py
```

Tweet tampil satu per satu; ENTER bila setuju usulan, ketik angka bila tidak.
Tersimpan **setiap baris** — boleh berhenti kapan saja dan dilanjutkan.

### Panduan Label

| | | |
|---|---|---|
| **0** | Negatif | keluhan, kecewa, kritik, rugi |
| **1** | Netral | pertanyaan, berita, info, **iklan/promo** |
| **2** | Positif | pujian, puas, terima kasih, rekomendasi |

- Tidak ada opini → **1**
- Menyebut Traveloka tapi bukan menilainya → **1**
- Sarkasme → ikuti **maksudnya**
- Ragu → **1**

Nilai **sikap terhadap Traveloka**, bukan suasana hati penulis.

In [ ]:
# Cek berapa yang sudah terisi
import pandas as pd, glob
p = sorted(glob.glob('data/gold/*.csv'))[-1]
d = pd.read_csv(p, encoding='utf-8-sig')
n = pd.to_numeric(d['label_manual'], errors='coerce').notna().sum()
print(f'{p}\n  {n}/{len(d)} terisi, sisa {len(d)-n}')

## 9. Cohen's Kappa — Validasi Pelabelan Otomatis

| Kappa | Arti | Tindakan |
|---|---|---|
| < 0,20 | sangat rendah | jangan pakai label otomatis |
| 0,20–0,40 | rendah | perbaiki lexicon/ambang dulu |
| 0,40–0,60 | sedang | boleh dipakai, sebutkan keterbatasannya |
| 0,60–0,80 | baik | aman |
| > 0,80 | sangat baik | aman |

**Laporkan berapa pun hasilnya.** Kappa rendah bukan kegagalan — itu temuan
jujur tentang keterbatasan pelabelan berbasis lexicon, dan justru menjadi
kontribusi penelitian.

In [ ]:
!python main.py --kappa

## 10. Bangun Dataset Berlabel Manusia

Bila Kappa rendah (< 0,40), label lexicon **tidak boleh** dipakai melatih
model. Sel ini menggabungkan label manual Anda dengan `stemmed_text` dari
hasil preprocessing.

> Hanya baris yang sudah Anda labeli yang ikut. Untuk memakai seluruh
> dataset, labeli semuanya lebih dulu di sel 8.

In [ ]:
import pandas as pd, glob

g   = pd.read_csv(sorted(glob.glob('data/gold/*.csv'))[-1], encoding='utf-8-sig')
lab = pd.read_csv(sorted(glob.glob('data/labeled/traveloka_*.csv'))[-1], encoding='utf-8-sig')

m = g[['id','label_manual']].merge(lab[['id','stemmed_text','username','date']], on='id')
m = m.rename(columns={'label_manual':'label'})
m['label'] = pd.to_numeric(m['label'], errors='coerce')
m = m.dropna(subset=['label','stemmed_text'])
m['label'] = m['label'].astype(int)
m = m[m['stemmed_text'].astype(str).str.strip() != '']

out = 'data/labeled/final_manual_labeled.csv'
m.to_csv(out, index=False, encoding='utf-8-sig')

print(f'Dataset berlabel manusia: {len(m)} dokumen -> {out}')
print(m['label'].value_counts().sort_index()
       .rename(index={0:'negatif',1:'netral',2:'positif'}).to_string())

## 11. Training & Evaluasi

1. Split 80/20 stratified — **data uji dibiarkan timpang**
2. Oversampling **hanya pada data latih** (mencegah data leakage)
3. Baseline kelas mayoritas sebagai acuan minimum
4. TF-IDF (unigram+bigram) → GridSearch alpha (`scoring=f1_macro`)
5. **MultinomialNB** — sesuai judul penelitian
6. CV 10-fold dengan oversampling di dalam tiap fold
7. Interval kepercayaan macro-F1 (bootstrap 2000×)

> Tambahkan `--compare` bila ingin ComplementNB sebagai pembanding
> (uji McNemar akan ikut dijalankan).

In [ ]:
!python main.py --train --labeled data/labeled/final_manual_labeled.csv

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display

m = sorted(glob.glob('results/metrics_*.csv'))
if m:
    display(pd.read_csv(m[-1], encoding='utf-8-sig').T)
for img in sorted(glob.glob('results/confusion_matrix_*.png'))[-2:]:
    print(img); display(Image(img))

---

## Yang Wajib Dilaporkan di Skripsi

1. Jumlah data tiap tahap penyaringan (mentah → bahasa → relevansi →
   korporat → per-penulis → final)
2. **Nilai Cohen's Kappa** beserta interpretasinya
3. Bagian *"Dilewati / gagal"* dari laporan scraping — keterbatasan
   pengumpulan data
4. Isi `lexicon/custom.tsv` — sebutkan sebagai *"InSet augmented with
   domain terms"*
5. Alasan operator `lang:id` tidak dipakai
6. **Macro-F1 dan baseline**, bukan accuracy saja
7. Interval kepercayaan tiap metrik

## Keterbatasan yang Wajib Diakui

- Pelabelan lexicon bukan ground truth (κ rendah)
- Bila memakai usulan pra-anotasi: sebut *"pelabelan manual dengan bantuan
  pra-anotasi otomatis"*
- Anotator tunggal — standar emas adalah ≥2 anotator independen
- Sampling tab *Latest* bersifat bias-kebaruan, bukan acak
- `langdetect` kurang akurat pada teks pendek
- X memotong tweet panjang
- Scraping melanggar ToS X; pertimbangkan anonimisasi kolom `username`
  pada lampiran

---

Dokumentasi lengkap — rumus, sumber data, dan pemetaan naskah:
[README di repositori](https://github.com/fahmikemal/traveloka-scraper)